# Autonomous Payment Agent with Claude and DPX

Builds a Claude agent that executes a cross-border vendor payment end-to-end — oracle gate → fee quote → compliance screen → settlement. No human step required.

**No wallet or real funds needed.** All DPX endpoints are free and require no authentication. Sandbox mode by default.

## Architecture
```
Claude (claude-sonnet-5)
  └── check_oracle_conditions  → GET  stability.untitledfinancial.com/reliability
  └── get_fee_quote            → GET  stability.untitledfinancial.com/quote
  └── screen_counterparty      → GET  agent.untitledfinancial.com/flow-check
  └── execute_settlement       → POST agent.untitledfinancial.com/settle
```

In [ ]:
%pip install anthropic httpx python-dotenv --quiet

In [ ]:
import os, json, httpx, anthropic
from dotenv import load_dotenv
load_dotenv()

client  = anthropic.Anthropic(api_key=os.environ.get('ANTHROPIC_API_KEY'))
SANDBOX = os.environ.get('SANDBOX', 'true').lower() != 'false'
ORACLE  = 'https://stability.untitledfinancial.com'
AGENT   = 'https://agent.untitledfinancial.com'

print(f'Sandbox: {SANDBOX}  |  Model: claude-sonnet-5')

## Tool schemas

In [ ]:
tools = [
  {
    'name': 'check_oracle_conditions',
    'description': 'Check global macro stability. Returns STABLE/CAUTION/UNSTABLE with score and reasoning. Always call first — abort if UNSTABLE.',
    'input_schema': {'type': 'object', 'properties': {}, 'required': []}
  },
  {
    'name': 'get_fee_quote',
    'description': 'Get a binding fee quote (valid 300s). Returns quote_id, fee breakdown, and net settlement amount.',
    'input_schema': {
      'type': 'object',
      'properties': {
        'amount_usd': {'type': 'number', 'description': 'Settlement amount in USD'},
        'has_fx':     {'type': 'boolean', 'description': 'True if cross-currency'},
        'esg_score':  {'type': 'number', 'description': 'Counterparty ESG score 0-100 (default 75)'}
      },
      'required': ['amount_usd', 'has_fx']
    }
  },
  {
    'name': 'screen_counterparty',
    'description': 'AML + sanctions + FATF R16 screen. Returns PROCEED, HOLD, or BLOCKED. Never proceed if BLOCKED.',
    'input_schema': {
      'type': 'object',
      'properties': {
        'amount':            {'type': 'number', 'description': 'Payment amount USD'},
        'recipient_address': {'type': 'string', 'description': 'Wallet address (0x...)'},
        'counterparty_name': {'type': 'string', 'description': 'Legal entity name'}
      },
      'required': ['amount', 'recipient_address', 'counterparty_name']
    }
  },
  {
    'name': 'execute_settlement',
    'description': 'Execute the settlement. Only call after oracle passes, quote obtained, and counterparty PROCEED.',
    'input_schema': {
      'type': 'object',
      'properties': {
        'amount':            {'type': 'number', 'description': 'Settlement amount USD'},
        'recipient_address': {'type': 'string', 'description': 'Wallet address'},
        'quote_id':          {'type': 'string', 'description': 'From get_fee_quote (valid 300s)'},
        'purpose':           {'type': 'string', 'description': 'e.g. vendor-invoice, payroll'}
      },
      'required': ['amount', 'recipient_address', 'quote_id', 'purpose']
    }
  }
]
print(f'{len(tools)} tools defined')

## Tool implementations

In [ ]:
def check_oracle_conditions():
    r = httpx.get(f'{ORACLE}/reliability', timeout=10).json()
    return {
        'status':    r.get('stability', {}).get('latestStatus', 'UNKNOWN'),
        'score':     r.get('stability', {}).get('currentScore', 0),
        'reasoning': r.get('intelligence', {}).get('reasoning', ''),
    }

def get_fee_quote(amount_usd, has_fx, esg_score=75):
    r = httpx.get(f'{ORACLE}/quote',
        params={'amountUsd': amount_usd, 'hasFx': str(has_fx).lower(), 'esgScore': esg_score},
        timeout=10).json()
    return {
        'quote_id':  r.get('quoteId'),
        'total_bps': r.get('fees', {}).get('total', {}).get('bps'),
        'fee_usd':   r.get('fees', {}).get('total', {}).get('usd'),
        'net_usd':   r.get('settlement', {}).get('netUsd'),
        'expires_in': '300s',
    }

def screen_counterparty(amount, recipient_address, counterparty_name):
    r = httpx.get(f'{AGENT}/flow-check',
        params={'amount': amount, 'from': 'USD', 'to': 'USD', 'address': recipient_address},
        timeout=15).json()
    compliance = r.get('compliance', {})
    return {
        'decision': r.get('decision', 'BLOCKED'),
        'checked':  compliance.get('checked', False),
        'verdict':  compliance.get('verdict', 'UNKNOWN'),
        'reason':   r.get('holdReason') or r.get('blockReason') or 'All checks passed',
    }

def execute_settlement(amount, recipient_address, quote_id, purpose):
    r = httpx.post(f'{AGENT}/settle', json={
        'amount': amount, 'sourceCurrency': 'USD', 'destinationCurrency': 'USD',
        'recipientAddress': recipient_address, 'purpose': purpose,
        'quoteId': quote_id, 'sandbox': SANDBOX,
    }, timeout=30).json()
    return {
        'status':        r.get('status'),
        'settlement_id': r.get('settlementId'),
        'tx_hash':       r.get('txHash'),
    }

def dispatch_tool(name, inputs):
    fns = {
        'check_oracle_conditions': check_oracle_conditions,
        'get_fee_quote':           get_fee_quote,
        'screen_counterparty':     screen_counterparty,
        'execute_settlement':      execute_settlement,
    }
    return json.dumps(fns[name](**inputs) if name in fns else {'error': f'unknown tool {name}'})

print('Tool implementations ready')

## Agent loop

In [ ]:
def run_payment_agent(task):
    print(f'\nTask: {task}\n' + '='*60)
    messages = [{'role': 'user', 'content': task}]
    system = (
        'You are an autonomous payment agent. For any payment request:\n'
        '1. check_oracle_conditions — abort if UNSTABLE\n'
        '2. get_fee_quote\n'
        '3. screen_counterparty — abort if BLOCKED, escalate if HOLD\n'
        '4. execute_settlement with the quote_id from step 2\n'
        'Explain your reasoning at each step.'
    )
    while True:
        resp = client.messages.create(
            model='claude-sonnet-5', max_tokens=4096,
            system=system, tools=tools, messages=messages,
        )
        for b in resp.content:
            if b.type == 'text' and b.text.strip():
                print(f'\nClaude: {b.text}')
        if resp.stop_reason == 'end_turn':
            return next((b.text for b in resp.content if b.type == 'text'), '')
        calls = [b for b in resp.content if b.type == 'tool_use']
        if not calls: break
        results = []
        for c in calls:
            print(f'\n  → {c.name}({json.dumps(c.input)})')
            out = dispatch_tool(c.name, c.input)
            print(f'  ← {out}')
            results.append({'type': 'tool_result', 'tool_use_id': c.id, 'content': out})
        messages += [{'role': 'assistant', 'content': resp.content}, {'role': 'user', 'content': results}]
    return ''

print('Agent ready')

## Run it

In [ ]:
result = run_payment_agent(
    'Pay $25,000 USD to 0xd8dA6BF26964aF9D7eEd9e03E53415D37aA96045 '
    'for Acme GmbH — vendor invoice #INV-2026-0042. Same-currency USD.'
)
print('\n' + '='*60 + '\nResult:', result)

## Screening a counterparty

`screen_counterparty` runs on every payment. The address below isn't on any
sanctions list, so this call will genuinely check it and come back clean
(`checked: true`, `verdict: UNKNOWN` — not flagged, not a known-bad match) —
that's a real pass, not a skipped check. A real sanctioned address would come
back `BLOCKED`, and the agent above is instructed to abort in that case.

In [ ]:
result = run_payment_agent(
    'Pay $10,000 USD to 0x1111111111111111111111111111111111111111 '
    'for Example Vendor LLC.'
)
print('Result:', result)

## Next steps

- [Multi-agent payments](https://docs.untitledfinancial.com/guides/multi-agent-payments) — orchestrator/sub-agent delegation with enforced spend limits
- [Agent frameworks](https://docs.untitledfinancial.com/guides/agent-frameworks) — same tools for OpenAI Agents SDK, smolagents, AutoGen, Google ADK
- **Go live**: set `SANDBOX=false` in `.env` and fund a Base wallet with USDC
- **Use MCP**: `npx @untitledfinancial/dpx-mcp` gives Claude Desktop all 83 DPX tools with no HTTP wiring